# GreenLawAI — LLM Server (Fixed)

## Before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Click the key icon (Secrets) → add `NGROK_TOKEN` from dashboard.ngrok.com
3. Runtime → Run all
4. Copy the URL printed in Cell 7 → paste into `.env` as `COLAB_URL=`

In [ ]:
# Cell 1 — Verify GPU
!nvidia-smi

In [ ]:
# Cell 2 — Install dependencies
!pip install -q transformers torch flask flask-cors pyngrok accelerate bitsandbytes

In [ ]:
# Cell 3 — Install ngrok binary manually (avoids 403 download error)
import os
BINARY_PATH = "/usr/local/bin/ngrok_binary"

if not os.path.exists(BINARY_PATH):
    print("Installing ngrok binary...")
    !wget -q -c -nc https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
    !tar -xzf ngrok-v3-stable-linux-amd64.tgz
    !mv ngrok {BINARY_PATH}
    !chmod +x {BINARY_PATH}
    !rm -f ngrok-v3-stable-linux-amd64.tgz
    print("ngrok installed to", BINARY_PATH)
else:
    print("ngrok already present")

In [ ]:
# Cell 4 — Imports
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok, conf
import threading
from google.colab import userdata

print(f"GPU available: {torch.cuda.is_available()}")

In [ ]:
# Cell 5 — Load model
MODEL_NAME = "microsoft/phi-2"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
config.pad_token_id = tokenizer.pad_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    config=config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded")

In [ ]:
# Cell 6 — Flask API
# IMPORTANT: This version accepts BOTH prompt= and context/question= formats
# so it works with the updated colab_llm_client.py

app = Flask(__name__)
CORS(app)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        "status": "active",
        "gpu": torch.cuda.is_available(),
        "model": MODEL_NAME
    })

@app.route('/generate', methods=['POST'])
def generate():
    try:
        data = request.json

        # Accept full prompt OR context+question split
        if data.get('prompt'):
            # New format: full prompt from LegalAgent
            prompt = data['prompt']
        else:
            # Legacy format: context + question
            context  = data.get('context', '')
            question = data.get('question', '')
            prompt   = f"Context: {context}\nQuestion: {question}\nAnswer:"

        max_new_tokens = data.get('max_new_tokens', data.get('max_tokens', 500))
        temperature    = float(data.get('temperature', 0.1))
        do_sample      = bool(data.get('do_sample', temperature > 0))

        inputs = tokenizer(
            prompt,
            return_tensors='pt',
            truncation=True,
            max_length=2048
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature if do_sample else 1.0,
                do_sample=do_sample,
                pad_token_id=tokenizer.pad_token_id,
                repetition_penalty=float(data.get('repetition_penalty', 1.3)),
            )

        # Decode only the NEW tokens (not the prompt)
        new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
        answer     = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        return jsonify({"answer": answer, "status": "success"})

    except Exception as e:
        return jsonify({"error": str(e), "status": "error"}), 500

print("API endpoints ready: /health  /generate")

In [ ]:
# Cell 7 — Start tunnel (HTTP mode — no SSL errors)
conf.get_default().ngrok_path = BINARY_PATH

TOKEN = userdata.get('NGROK_TOKEN')
!{BINARY_PATH} config add-authtoken {TOKEN}

# bind_tls=False → HTTP tunnel → no SSLEOFError on client side
tunnel     = ngrok.connect(5000, bind_tls=False)
public_url = tunnel.public_url

print("=" * 60)
print("SERVER URL (copy this into your .env)")
print("=" * 60)
print(f"\nCOLAB_URL={public_url}\n")
print("NOTE: URL starts with http:// not https://")
print("NOTE: Free ngrok URLs change every session — update .env each time")
print("=" * 60)

In [ ]:
# Cell 8 — Start Flask server
def start_server():
    app.run(port=5000, use_reloader=False)

threading.Thread(target=start_server, daemon=True).start()
print("Server running on port 5000")
print("Public URL:", public_url)